# Frameworks 01 - OpenAI Agents

Objetivo: ejecutar un Agent real mediante openai-agents sobre el Provider
seleccionado por .env, exigir evidencia de una Tool y producir una respuesta
humana. El mismo notebook conserva un control determinista offline para CI.

**Lugar en el modelo:** OpenAI Agents controla Runner, sessions, guardrails y
handoffs; el Provider controla la inferencia. Son decisiones independientes.

**Evidencia exigida:** human_result debe registrar Provider, Framework,
modelo, Tool, respuesta pública y linaje. Una respuesta JSON o
python-runtime sólo es válida cuando el preflight declara explícitamente
offline-deterministic-control.

**Límite de la evidencia:** el control offline demuestra integración del SDK, no inferencia LM; el modo live debe conservar Provider, Framework, Tool y respuesta humana observados.


## Parametros de la demostracion

.env es la fuente canónica. AGENTIC_SYSTEMS_PROVIDER puede fijar el Provider
o dejar auto; el flag live del Provider seleccionado decide si se invoca el
modelo. RUN_OPENAI_AGENTS_LIVE es únicamente un override opcional.

| Variable | Default | Propósito |
|---|---|---|
| AGENTIC_SYSTEMS_PROVIDER | auto | Provider para este Framework. |
| RUN_OPENAI_AGENTS_LIVE | flag del Provider | Override opcional de ejecución live. |
| RUN_OPENAI_LIVE / RUN_BEDROCK_LIVE / RUN_OLLAMA_LIVE / RUN_VLLM_LIVE | .env | Autoriza la llamada real correspondiente. |
| Framework | openai-agents | Runner, session, guardrail y handoff. |


## 1) Frontera Provider x Framework


In [ ]:
import importlib.util
import os

import agentic_systems as toolkit

os.environ.setdefault("OPENAI_AGENTS_DISABLE_TRACING", "1")

# OPENAI_AGENTS_DEPENDENCY: resolve availability and installation from the registry.
framework_name = "openai-agents"
install_target = toolkit.dependency_target(framework_name, kind="framework")
if install_target is None:
    raise RuntimeError(f"{framework_name!r} has no registered install target.")
if importlib.util.find_spec("agents") is None:
    get_ipython().run_line_magic("pip", f'install -q "{install_target}"')
importlib.invalidate_caches()
if importlib.util.find_spec("agents") is None:
    raise ImportError(
        f"{framework_name!r} remains unavailable after installing "
        f'"{install_target}". Restart this kernel and run the notebook again.'
    )

from agents import (  # noqa: E402
    GuardrailFunctionOutput,
    SQLiteSession,
    handoff,
    input_guardrail,
)


def _enabled(value: str | None) -> bool:
    return str(value or "").strip().lower() in {"1", "true", "yes"}


requested_provider = os.getenv("AGENTIC_SYSTEMS_PROVIDER", "auto").strip() or "auto"
candidate_runtime = toolkit.runtime(provider=requested_provider)
candidate_description = candidate_runtime.describe()
selected_provider = candidate_description["selected_provider"]
provider_live_flag = (
    f"RUN_{selected_provider.removesuffix('-runtime').replace('-', '_').upper()}_LIVE"
    if selected_provider and selected_provider != "python-runtime"
    else None
)
provider_live_value = os.getenv(provider_live_flag, "0") if provider_live_flag else "0"
RUN_LIVE = (
    selected_provider != "python-runtime"
    and _enabled(os.getenv("RUN_OPENAI_AGENTS_LIVE", provider_live_value))
)

runtime = (
    candidate_runtime
    if RUN_LIVE
    else toolkit.runtime(provider="python-runtime")
)
runtime_description = runtime.describe()
execution_kind = (
    "live-language-model"
    if RUN_LIVE
    else "offline-deterministic-control"
)
profile = toolkit.integrations.framework_profile("openai-agents")
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "requested_provider": requested_provider,
        "provider_live_flag": provider_live_flag,
        "provider_live_authorized": RUN_LIVE,
        "runtime": runtime_description,
        "profile": profile.to_dict(),
    },
    title="Provider x OpenAI Agents preflight",
)


## 2) Tool canónica, respuesta humana y session nativa


In [ ]:
@toolkit.tool
def inspect_public_api(symbol: str) -> dict:
    return {"symbol": symbol, "is_public": symbol in toolkit.__all__}


@input_guardrail(run_in_parallel=False)
def reject_blocked_input(context, native_agent, input_value):
    return GuardrailFunctionOutput(
        output_info={"checked": True},
        tripwire_triggered="blocked" in str(input_value).lower(),
    )


session = SQLiteSession("agentic-systems-tutorial", ":memory:")
framework = toolkit.framework(
    "openai-agents",
    agent_kwargs={"input_guardrails": [reject_blocked_input]},
    run_kwargs={"session": session},
)
agent = toolkit.agent(
    name="openai_agents_inspector",
    instructions=(
        "Usa inspect_public_api para verificar el símbolo solicitado. "
        "Después responde una sola frase natural en español; no devuelvas JSON."
    ),
    runtime=runtime,
    tools=[inspect_public_api],
    framework=framework,
    contract=toolkit.AgentContract(must_call=["inspect_public_api"]),
    policy=toolkit.RunPolicy(max_turns=4, max_tool_calls=1),
)
agent.prepare()

request = (
    "Verifica si framework pertenece a la API pública instalada."
    if RUN_LIVE
    else {"tool": "inspect_public_api", "input": {"symbol": "framework"}}
)
result = await agent.arun(request, mode="eval")

assert result.ok, result.errors
assert result.engine == runtime_description["selected_provider"]
assert result.meta["framework_adapter"] == "openai-agents"
assert any(
    event.name == "inspect_public_api" and event.ok
    for event in result.tool_events
)
result.raise_if_inconsistent()
assert not result.meta.get("fallback_provider")
if RUN_LIVE:
    assert result.engine != "python-runtime"
    assert result.text and not result.text.lstrip().startswith("{"), result.text

title = (
    "OpenAI Agents live RunResult"
    if RUN_LIVE
    else "OpenAI Agents offline deterministic control"
)
toolkit.human_result(result, title=title, show_lineage=True)
toolkit.show_json(
    {
        "execution_kind": execution_kind,
        "native_agent": type(agent.native_agent).__name__,
        "native_result": type(result.native_result).__name__,
        "framework_config": agent.framework_config.inspect(),
    },
    title="Runner evidence",
)


## 3) Guardrail operativo


In [ ]:
blocked_result = agent.run("blocked input", mode="eval")
assert not blocked_result.ok
session.close()
toolkit.show_json(
    {"ok": blocked_result.ok, "error": blocked_result.data["error"]["code"]},
    title="Input guardrail",
)


## 4) Handoff nativo explícito


In [ ]:
specialist = toolkit.agent(
    name="specialist",
    instructions="Confirma de forma natural que recibiste el handoff.",
    runtime=runtime,
    tools=[inspect_public_api],
    framework="openai-agents",
)
specialist.prepare()
native_handoff = handoff(specialist.native_agent)

handoff_session = SQLiteSession("handoff-tutorial", ":memory:")
triage = toolkit.agent(
    name="triage",
    instructions="Transfiere siempre la solicitud al especialista.",
    runtime=runtime,
    framework=toolkit.framework(
        "openai-agents",
        agent_kwargs={"handoffs": [native_handoff]},
        run_kwargs={"session": handoff_session},
    ),
)
handoff_result = triage.run(
    {"tool": native_handoff.tool_name, "input": {}},
    mode="eval",
)
assert handoff_result.ok
assert handoff_result.engine == runtime_description["selected_provider"]
assert handoff_result.meta["framework_adapter"] == "openai-agents"
assert not handoff_result.meta.get("fallback_provider")
last_agent = getattr(handoff_result.native_result, "last_agent", None)
last_agent_name = getattr(last_agent, "name", None)
handoff_certificate = {
    "schema_version": "agentic_systems.handoff-certificate.v1",
    "ok": last_agent_name == "specialist",
    "expected_agent": "specialist",
    "observed_agent": last_agent_name,
    "provider": handoff_result.engine,
    "framework": handoff_result.meta["framework_adapter"],
    "fallback_provider": handoff_result.meta.get("fallback_provider"),
}
assert handoff_certificate["ok"], handoff_certificate
if not RUN_LIVE:
    assert handoff_result.data == {"assistant": "specialist"}, handoff_result.data
handoff_session.close()
toolkit.show_json(
    {
        "handoff_tool": native_handoff.tool_name,
        "certificate": handoff_certificate,
        "final_output": handoff_result.data,
        "native_result": type(handoff_result.native_result).__name__,
    },
    title="Native handoff",
)


## 5) API realmente ejercitada


In [ ]:
api_coverage = [
    "toolkit.dependency_target", "toolkit.runtime", "toolkit.framework",
    "toolkit.integrations.framework_profile",
    "toolkit.tool", "toolkit.agent", "Agent.prepare", "Agent.native_agent",
    "agent.run", "agent.arun", "RunResult.native_result", "toolkit.RunPolicy",
    "toolkit.AgentContract", "toolkit.human_result", "toolkit.show_json",
]
toolkit.show_json(api_coverage, title="OpenAI Agents API coverage")


## Resultado e interpretacion

Cuando el preflight declara live-language-model, el Provider seleccionado por
.env realiza la inferencia y OpenAI Agents controla el loop, la Tool y la
session. human_result debe contener una frase humana y el linaje de la Tool.

Cuando declara offline-deterministic-control, python-runtime sólo verifica
la integración del SDK sin presentarse como un modelo de lenguaje.
